In [ ]:
!pip install pymeshlab

In [ ]:

import open3d as o3d
import numpy as np
# Load meshes
recon_mesh_path = "/home/user/menon/workspace/ros1/menon25iros_ws/src/kimera_semantics_evidential/kimera_semantics_evidential_ros/mesh_results/video_nyu13_swinmtl.ply"
gt_mesh_path = "/home/user/menon/workspace/ros1/menon25iros_ws/src/evaluation_3d/new_plys/scene00/scene0369_02_open3d_color.ply"
cleaned_mesh_path = "/home/user/menon/workspace/ros1/menon25iros_ws/src/kimera_semantics_evidential/kimera_semantics_evidential_ros/mesh_results/video_nyu13_swinmtl_cleaned_mesh.ply"


# Load ground truth and reconstructed mesh
groundtruth_mesh = o3d.io.read_triangle_mesh(gt_mesh_path)
reconstructed_mesh = o3d.io.read_triangle_mesh(recon_mesh_path)
print("Loaded meshes")

# Ensure the meshes have vertex normals for proper processing
groundtruth_mesh.compute_vertex_normals()
reconstructed_mesh.compute_vertex_normals()
print("Computed vertex normals of meshes")

if reconstructed_mesh.has_vertex_colors():
    colors_np = np.asarray(reconstructed_mesh.vertex_colors)
    print("Loaded vertex colors")
else:
    print("No vertex colors found")
    colors_np = None  # No colors available



# Convert the reconstructed mesh's vertices to a point cloud
reconstructed_pcd = o3d.geometry.PointCloud()
reconstructed_pcd.points = reconstructed_mesh.vertices  # Use **ALL** vertices, no downsampling

# Convert the ground truth mesh's vertices to a point cloud
groundtruth_pcd = o3d.geometry.PointCloud()
groundtruth_pcd.points = groundtruth_mesh.vertices  # Use **ALL** ground truth vertices

# Compute distance of each reconstructed vertex from the nearest ground truth vertex
distances = np.asarray(reconstructed_pcd.compute_point_cloud_distance(groundtruth_pcd))

# Verify correct array sizes
print(f"Reconstructed mesh vertices: {len(reconstructed_mesh.vertices)}")
print(f"Distance array length: {len(distances)}")

# Ensure distances array matches the number of vertices
assert len(distances) == len(reconstructed_mesh.vertices), "Mismatch in distance array length!"

# Set a threshold to filter distant vertices
distance_threshold = 0.3  # Adjust based on noise level
valid_vertex_mask = distances < distance_threshold

# Convert Open3D vector to NumPy array for filtering
vertices_np = np.asarray(reconstructed_mesh.vertices)

# Apply filtering based on valid_vertex_mask
filtered_vertices = vertices_np[valid_vertex_mask]
filtered_colors = colors_np[valid_vertex_mask] if colors_np is not None else None

# Create a mapping from old vertex indices to new indices
old_to_new_index = {old_idx: new_idx for new_idx, old_idx in enumerate(np.where(valid_vertex_mask)[0])}

# Filter faces: Keep only faces where all three vertices exist in the new set
filtered_faces = []
for face in np.asarray(reconstructed_mesh.triangles):
    if all(v in old_to_new_index for v in face):
        new_face = [old_to_new_index[v] for v in face]
        filtered_faces.append(new_face)

# Create the cleaned mesh
cleaned_mesh = o3d.geometry.TriangleMesh()
cleaned_mesh.vertices = o3d.utility.Vector3dVector(filtered_vertices)
cleaned_mesh.triangles = o3d.utility.Vector3iVector(np.array(filtered_faces))

if filtered_colors is not None:
    cleaned_mesh.vertex_colors = o3d.utility.Vector3dVector(filtered_colors)

# Apply Laplacian smoothing
#cleaned_mesh = cleaned_mesh.filter_smooth_laplacian(number_of_iterations=1)
# Save the cleaned and smoothed mesh
o3d.io.write_triangle_mesh(cleaned_mesh_path, cleaned_mesh)
print("Saved cleaned mesh")

# Visualize the result
o3d.visualization.draw_geometries([cleaned_mesh])


Loaded meshes
Computed vertex normals of meshes
Loaded vertex colors
Reconstructed mesh vertices: 483981
Distance array length: 483981
